# Fase 1 · M05/M06: Grafos de Trazabilidad

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 1 — Transformación |
| **Módulo** | M05/M06 — Trazabilidad |

---

## 🎯 Qué hace

Genera grafos de trazabilidad del pipeline de transformación mostrando el flujo de datos desde las tablas originales hasta df_alumno.

## 📋 Requisitos

- `data/01_interim/*.parquet`
- `data/02_processed/df_alumno.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `results/fase1/trazabilidad_fase1.png` | Grafo de trazabilidad |

## 🔄 Flujo

```
df_alumno.parquet + interim/*.parquet
    ↓ Construcción del grafo de dependencias
    → results/fase1/trazabilidad_fase1.png
```

## ➡️ Siguiente

`f1_m06_grafo.ipynb` — versión interactiva con Pyvis


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
#
# Importa todo lo necesario para los 4 artefactos:
#   - matplotlib: imagen estática del grafo
#   - pyvis: grafo interactivo con física
#   - python-docx: documento Word
#   - src.html: navegación y plantillas HTML
# ============================================================================

import sys
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# --- Detectar entorno ---
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

if not (ROOT / 'src').exists():
    raise FileNotFoundError(
        f'No se encontró la carpeta src/ en {ROOT}\n'
        f'Asegúrate de que este notebook está dentro del proyecto (AU_UJI/)'
    )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
from src.config import RUTA_HTML, RUTA_INTERIM, RUTA_PROCESSED, info_entorno
from src.utils import formato_numero_es
from src.constantes import TABLA_PREINSCRIPCION
from src.html import generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

# --- Imports externos (pueden necesitar pip install) ---
try:
    from pyvis.network import Network
    HAS_PYVIS = True
except ImportError:
    HAS_PYVIS = False
    print('⚠️ pyvis no instalado — se omitirá M06. Instalar: pip install pyvis')

try:
    from docx import Document
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    HAS_DOCX = True
except ImportError:
    HAS_DOCX = False
    print('⚠️ python-docx no instalado — se omitirá Word. Instalar: pip install python-docx')

# --- Atajo ---
fmt = formato_numero_es

# --- Carpetas de salida ---
RUTA_FASE1 = RUTA_HTML / 'fase1'
RUTA_FASE1.mkdir(parents=True, exist_ok=True)
RUTA_DOCS = ROOT / 'docs'
RUTA_DOCS.mkdir(parents=True, exist_ok=True)

info_entorno()

⚠️ python-docx no instalado — se omitirá Word. Instalar: pip install python-docx
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_

In [2]:
# ============================================================================
# CELDA 2: LEER DATOS
# ============================================================================
#
# Carga métricas de todas las tablas + dataset final.
# Se separan por origen (Excel principal vs Preinscripción) para
# mostrar líneas sólidas vs punteadas en los grafos.
# ============================================================================

print('=' * 60)
print('LEYENDO PARQUETS')
print('=' * 60)

TABLAS = {}
for pq in sorted(RUTA_INTERIM.glob('*.parquet')):
    df = pd.read_parquet(pq)
    TABLAS[pq.stem] = {
        'filas': len(df),
        'columnas': list(df.columns),
        'n_cols': len(df.columns),
        'es_preins': pq.stem == TABLA_PREINSCRIPCION
    }
    print(f'  ✅ {pq.stem:15s}: {fmt(len(df)):>10s} × {len(df.columns)} cols')

# Dataset final (dinámico, sin hardcodes)
df_path = RUTA_PROCESSED / 'df_alumno.parquet'
if df_path.exists():
    _df = pd.read_parquet(df_path)
    DF_INFO = {
        'filas': len(_df),
        'n_cols': len(_df.columns),
        'columnas': list(_df.columns)
    }
    del _df  # Liberar memoria
else:
    raise FileNotFoundError(
        f'No encontrado: {df_path}\n'
        f'Ejecuta primero f1_m04_dataset_final.ipynb'
    )

print(f'\n🎯 df_alumno: {fmt(DF_INFO["filas"])} × {DF_INFO["n_cols"]} cols')

# Separar por origen
tabs_datos = sorted([t for t in TABLAS if not TABLAS[t]['es_preins']])
tabs_preinsc = [t for t in TABLAS if TABLAS[t]['es_preins']]

LEYENDO PARQUETS
  ✅ becas          :     70.524 × 4 cols
  ✅ demograficos   :     30.873 × 5 cols


  ✅ domicilios     :    210.911 × 6 cols


  ✅ expedientes    :    109.568 × 15 cols
  ✅ notas          :    107.908 × 5 cols


  ✅ preinscripcion :    210.986 × 24 cols
  ✅ recibos        :    114.454 × 5 cols
  ✅ titulaciones   :         45 × 6 cols
  ✅ trabajo        :    195.524 × 4 cols



🎯 df_alumno: 109.568 × 37 cols


In [3]:
# ============================================================================
# CELDA 3: IMAGEN MATPLOTLIB
# ============================================================================
#
# Genera un grafo estático con matplotlib:
#   - Recuadro izquierdo: 8 tablas del Excel principal (azul)
#   - Recuadro inferior: preinscripción (morado, línea punteada)
#   - Recuadro derecho: df_alumno (verde, destino)
#   - Flechas Bézier conectando cada tabla al destino
#
# Se guarda como JPG (para documentos) y PNG (para web).
# ============================================================================

print('=' * 60)
print('GENERANDO IMAGEN')
print('=' * 60)

fig, ax = plt.subplots(figsize=(16, 11), facecolor='#f7fafc')
ax.set_facecolor('#f7fafc')
ax.set_xlim(0, 16)
ax.set_ylim(0, 11)
ax.axis('off')

ys = np.linspace(10, 1.5, len(tabs_datos))

# Recuadro Excel principal
ax.add_patch(FancyBboxPatch(
    (0.5, 1), 3.5, 9.5,
    boxstyle='round,rounding_size=0.3',
    fc='#edf2f7', ec='#a0aec0', lw=2, ls='--', alpha=0.6
))
ax.text(2.25, 10.7, 'datos_proyecto_sin_preinscrip.xlsx',
        ha='center', fontsize=10, color='#4a5568', fontweight='bold')

# Tablas del Excel principal
for i, t in enumerate(tabs_datos):
    y = ys[i]
    ax.add_patch(FancyBboxPatch(
        (0.8, y - 0.25), 2.9, 0.55,
        boxstyle='round,rounding_size=0.12',
        fc='#3182ce', ec='#2c5282', lw=1.5
    ))
    ax.text(2.25, y + 0.08, t, ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')
    ax.text(2.25, y - 0.15, f'({TABLAS[t]["n_cols"]} cols)',
            ha='center', va='center', fontsize=9, color=(1, 1, 1, 0.8))
    ax.annotate('', xy=(11.5, 5.5), xytext=(3.8, y),
                arrowprops=dict(arrowstyle='->', color='#3182ce',
                                lw=1.8, connectionstyle='arc3,rad=0.08'))

# Preinscripción (punteada)
if tabs_preinsc:
    ax.add_patch(FancyBboxPatch(
        (5.5, 0.3), 4, 2,
        boxstyle='round,rounding_size=0.3',
        fc='#faf5ff', ec='#9f7aea', lw=2, ls='--', alpha=0.6
    ))
    ax.text(7.5, 2.5, 'preinscripcion_si.xlsx',
            ha='center', fontsize=10, color='#6b46c1', fontweight='bold')
    t = tabs_preinsc[0]
    ax.add_patch(FancyBboxPatch(
        (5.8, 0.7), 3.4, 1.2,
        boxstyle='round,rounding_size=0.15',
        fc='#805ad5', ec='#6b46c1', lw=1.5
    ))
    ax.text(7.5, 1.45, t, ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')
    ax.text(7.5, 1.05, f'({TABLAS[t]["n_cols"]} cols)',
            ha='center', va='center', fontsize=9, color=(1, 1, 1, 0.8))
    ax.annotate('', xy=(11.5, 4.8), xytext=(9.3, 1.3),
                arrowprops=dict(arrowstyle='->', color='#805ad5',
                                lw=2.5, connectionstyle='arc3,rad=-0.15', ls='--'))

# df_alumno (destino)
ax.add_patch(FancyBboxPatch(
    (11.5, 4), 3.5, 3,
    boxstyle='round,rounding_size=0.2',
    fc='#38a169', ec='#276749', lw=2.5
))
ax.text(13.25, 5.8, 'df_alumno', ha='center', va='center',
        fontsize=16, color='white', fontweight='bold')
ax.text(13.25, 5.2,
        f'({DF_INFO["n_cols"]} cols | {fmt(DF_INFO["filas"])})',
        ha='center', va='center', fontsize=11, color=(1, 1, 1, 0.9))

# Título
ax.text(8, 10.8, 'TFM: Predicción de Abandono Universitario',
        ha='center', fontsize=18, fontweight='bold', color='#1a202c')
ax.text(8, 10.35, f'Trazabilidad - Fase 1: {len(TABLAS)} tablas',
        ha='center', fontsize=12, color='#718096')

plt.tight_layout()
plt.savefig(RUTA_FASE1 / 'trazabilidad_fase1.jpg',
            dpi=150, bbox_inches='tight', facecolor='#f7fafc')
plt.savefig(RUTA_FASE1 / 'trazabilidad_fase1.png',
            dpi=150, bbox_inches='tight', facecolor='#f7fafc')
plt.close()
print('✅ trazabilidad_fase1.jpg/png')

GENERANDO IMAGEN


✅ trazabilidad_fase1.jpg/png


In [4]:
# ============================================================================
# CELDA 4: GRAFO D3.js (m05_grafo_d3.html)
# ============================================================================
#
# Genera un grafo interactivo con D3.js (cargado desde CDN).
# Cada nodo es clicable: muestra las columnas en un panel lateral.
# Las posiciones se calculan dinámicamente según el nº de tablas.
#
# Usa render_base_html() para mantener la plantilla y navegación
# consistentes con el resto del proyecto.
# ============================================================================

print('=' * 60)
print('GENERANDO M05: GRAFO D3')
print('=' * 60)

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase1', modulo_activo='m05'
)

# --- Generar datos JS dinámicamente ---
tabs_js = []
for t in tabs_datos:
    cols_json = str(TABLAS[t]['columnas']).replace(chr(39), chr(34))
    tabs_js.append(
        f"{{id:'{t}',cols:{TABLAS[t]['n_cols']},"
        f"rows:{TABLAS[t]['filas']},color:'#3182ce',"
        f"columns:{cols_json}}}"
    )
for t in tabs_preinsc:
    cols_json = str(TABLAS[t]['columnas']).replace(chr(39), chr(34))
    tabs_js.append(
        f"{{id:'{t}',cols:{TABLAS[t]['n_cols']},"
        f"rows:{TABLAS[t]['filas']},color:'#805ad5',"
        f"columns:{cols_json},future:true}}"
    )
tabs_js.append(
    f"{{id:'df_alumno',cols:{DF_INFO['n_cols']},"
    f"rows:{DF_INFO['filas']},color:'#38a169',target:true}}"
)

# Posiciones dinámicas
n = len(tabs_datos)
pos_js = [f'{{x:70,y:{35 + i * 55}}}' for i in range(n)]
pos_js.append(f'{{x:280,y:{35 + n * 55}}}')
pos_js.append(f'{{x:700,y:{(n * 55) // 2 + 35}}}')

svg_height = max(n * 55 + 150, 550)

# --- HTML con D3 ---
contenido_d3 = f'''
<div class="grafo-leyenda">
    <div class="leyenda-item"><div class="leyenda-color" style="background:#3182ce"></div>datos_proyecto ({len(tabs_datos)})</div>
    <div class="leyenda-item"><div class="leyenda-color" style="background:#805ad5"></div>preinscripcion ({len(tabs_preinsc)})</div>
    <div class="leyenda-item"><div class="leyenda-color" style="background:#38a169"></div>df_alumno</div>
    <div style="margin-left:auto;color:#718096;font-size:0.85em">👆 Clic en tabla para ver columnas</div>
</div>

<div style="display:grid;grid-template-columns:1fr 350px;gap:20px;margin-top:15px;">
    <div style="background:white;border-radius:12px;padding:20px;box-shadow:0 2px 4px rgba(0,0,0,0.08);">
        <svg id="diagram" style="width:100%;height:{svg_height}px;"></svg>
    </div>
    <div id="infoPanel" style="background:white;border-radius:12px;padding:20px;box-shadow:0 2px 4px rgba(0,0,0,0.08);">
        <div style="text-align:center;color:#718096;padding:40px">
            <p style="font-size:48px">👆</p>
            <p>Clic en una tabla</p>
        </div>
    </div>
</div>
<div class="tooltip" id="tooltip" style="position:absolute;background:#2d3748;color:white;padding:10px;border-radius:8px;font-size:12px;opacity:0;pointer-events:none;z-index:1000;"></div>
'''

# Script D3
script_d3 = f'''
const tabs = [{','.join(tabs_js)}];
const pos = [{','.join(pos_js)}];
tabs.forEach((t,i) => {{ t.x = pos[i].x; t.y = pos[i].y; }});

const svg = d3.select("#diagram");
const tooltip = d3.select("#tooltip");
const target = tabs.find(t => t.target);

tabs.forEach(t => {{
    if (!t.target) svg.append("path")
        .attr("class", `link ${{t.future ? 'future' : ''}}`)
        .attr("stroke", t.future ? '#ed8936' : t.color)
        .attr("fill", "none")
        .attr("stroke-width", 2)
        .attr("d", `M ${{t.x+115}} ${{t.y+25}} C ${{t.x+250}} ${{t.y+25}}, ${{target.x-100}} ${{target.y+35}}, ${{target.x}} ${{target.y+35}}`);
}});

tabs.forEach(t => {{
    const g = svg.append("g")
        .attr("class", "node")
        .attr("transform", `translate(${{t.x}},${{t.y}})`)
        .style("cursor", "pointer")
        .on("click", () => showInfo(t))
        .on("mouseover", e => {{
            tooltip.html(`<strong>${{t.id}}</strong><br>${{t.cols}} cols`)
                .style("left", (e.pageX+10)+"px")
                .style("top", (e.pageY-10)+"px")
                .style("opacity", 1);
        }})
        .on("mouseout", () => tooltip.style("opacity", 0));

    const w = t.target ? 130 : 115, h = t.target ? 70 : 50;
    g.append("rect").attr("width", w).attr("height", h)
        .attr("fill", t.color).attr("rx", 8)
        .attr("stroke", "#2d3748").attr("stroke-width", 1.5);
    g.append("text").attr("x", w/2).attr("y", t.target ? 28 : 20)
        .attr("text-anchor", "middle").attr("fill", "white")
        .attr("font-weight", "bold").attr("font-size", "12px").text(t.id);
    g.append("text").attr("x", w/2).attr("y", t.target ? 45 : 35)
        .attr("text-anchor", "middle").attr("fill", "rgba(255,255,255,0.8)")
        .attr("font-size", "10px").text(`${{t.cols}} cols`);
}});

function showInfo(t) {{
    let html = `<h3 style="color:#3182ce;margin-bottom:15px;border-bottom:2px solid #3182ce;padding-bottom:10px">📋 ${{t.id}}</h3>`;
    html += `<p style="color:#718096;margin-bottom:15px">${{t.cols}} columnas | ${{t.rows?.toLocaleString('es-ES') || ''}} filas</p>`;
    if (t.columns && t.columns.length > 0) {{
        html += `<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(140px,1fr));gap:6px;">`;
        t.columns.forEach(c => html += `<div style="background:#f7fafc;padding:6px 10px;border-radius:6px;border-left:3px solid #3182ce;font-size:11px;font-family:monospace;">${{c}}</div>`);
        html += `</div>`;
    }}
    document.getElementById("infoPanel").innerHTML = html;
}}
'''

# Estilos adicionales
estilos_d3 = '''
<style>
.link.future { stroke-dasharray: 8,4; }
@media (max-width: 1000px) {
    .grafo-container { grid-template-columns: 1fr; }
}
</style>
'''

# Render con plantilla base
html_d3 = render_base_html(
    titulo='M05: Grafo D3',
    icono='📊',
    subtitulo='Trazabilidad de Columnas - Fase 1',
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=contenido_d3,
    estilos_adicionales=estilos_d3,
    scripts_adicionales=f'<script src="https://d3js.org/d3.v7.min.js"></script>\n<script>{script_d3}</script>',
    notebook_nombre='f1_m05_m06_trazabilidad.ipynb',
    notebook_carpeta='fase1_transformacion'
)

guardar_html(html_d3, RUTA_FASE1 / 'm05_grafo_d3.html')
print('✅ m05_grafo_d3.html')

GENERANDO M05: GRAFO D3
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m05_grafo_d3.html
✅ m05_grafo_d3.html


In [5]:
# ============================================================================
# CELDA 5: GRAFO PYVIS (m06_grafo_pyvis.html)
# ============================================================================
#
# PyVis genera un grafo con vis.js — tiene drag, zoom y hover nativos.
# Extraemos el <body> del HTML temporal y lo envolvemos con nuestra
# plantilla base para mantener la navegación del proyecto.
# ============================================================================

if HAS_PYVIS:
    print('=' * 60)
    print('GENERANDO M06: GRAFO PYVIS')
    print('=' * 60)

    net = Network(height='600px', width='100%', bgcolor='#f7fafc', directed=True)
    net.set_options('{"physics":{"enabled":false},"interaction":{"hover":true}}')

    # Nodos: tablas Excel principal
    for i, t in enumerate(tabs_datos):
        info = TABLAS[t]
        tip = (f"{fmt(info['filas'])} filas | {info['n_cols']} cols\n\n"
               f"Columnas:\n" + "\n".join([f"• {c}" for c in info['columnas']]))
        net.add_node(
            t, label=f"{t}\n({info['n_cols']} cols)",
            color='#3182ce', shape='box', title=tip,
            x=-400, y=-250 + i * 80, physics=False,
            font={'color': 'white', 'size': 12}
        )

    # Nodos: preinscripción
    for t in tabs_preinsc:
        info = TABLAS[t]
        tip = f"{fmt(info['filas'])} filas | {info['n_cols']} cols"
        net.add_node(
            t, label=f"{t}\n({info['n_cols']} cols)",
            color='#805ad5', shape='box', title=tip,
            x=-100, y=400, physics=False,
            font={'color': 'white', 'size': 12}
        )

    # Nodo destino
    net.add_node(
        'df_alumno',
        label=f"df_alumno\n({DF_INFO['n_cols']} cols | {fmt(DF_INFO['filas'])})",
        color='#38a169', shape='box',
        title=f"{fmt(DF_INFO['filas'])} filas",
        x=250, y=100, physics=False,
        font={'color': 'white', 'size': 14}
    )

    # Conexiones
    for t in tabs_datos:
        net.add_edge(t, 'df_alumno', color='#3182ce', width=2,
                     title=f"{TABLAS[t]['n_cols']} cols")
    for t in tabs_preinsc:
        net.add_edge(t, 'df_alumno', color='#ed8936', width=1.5,
                     dashes=True, title='Preinscripción')

    # Generar HTML temporal y extraer body
    temp = RUTA_FASE1 / '_temp_pyvis.html'
    net.save_graph(str(temp))
    with open(temp, 'r', encoding='utf-8') as f:
        pyvis_html = f.read()

    match = re.search(r'<body>(.*?)</body>', pyvis_html, re.DOTALL)
    body = match.group(1) if match else pyvis_html

    # Envolver con plantilla base
    nav_fases_m06, nav_modulos_m06 = generar_html_navegacion_completa(
        fase_activa='fase1', modulo_activo='m06'
    )

    contenido_pyvis = f'''
    <div class="grafo-leyenda">
        <div class="leyenda-item"><div class="leyenda-color" style="background:#3182ce"></div>datos_proyecto ({len(tabs_datos)})</div>
        <div class="leyenda-item"><div class="leyenda-color" style="background:#805ad5"></div>preinscripcion ({len(tabs_preinsc)})</div>
        <div class="leyenda-item"><div class="leyenda-color" style="background:#38a169"></div>df_alumno</div>
        <div style="margin-left:auto;color:#718096;font-size:0.85em">🖱️ Arrastra | 🔍 Zoom | 👆 Hover</div>
    </div>
    {body}
    '''

    estilos_pyvis = '''
    <style>
    #mynetwork { width: 100%; height: calc(100vh - 220px); min-height: 500px; background: #f7fafc; }
    </style>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.2/dist/dist/vis-network.min.css">
    <script src="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.2/dist/vis-network.min.js"></script>
    '''

    html_pyvis = render_base_html(
        titulo='M06: Grafo PyVis',
        icono='🕸️',
        subtitulo='Trazabilidad Interactiva - Fase 1',
        nav_fases=nav_fases_m06,
        nav_modulos=nav_modulos_m06,
        contenido=contenido_pyvis,
        estilos_adicionales=estilos_pyvis,
        notebook_nombre='f1_m05_m06_trazabilidad.ipynb',
        notebook_carpeta='fase1_transformacion'
    )

    guardar_html(html_pyvis, RUTA_FASE1 / 'm06_grafo_pyvis.html')
    temp.unlink()
    print('✅ m06_grafo_pyvis.html')
else:
    print('⚠️ PyVis no disponible — m06_grafo_pyvis.html omitido')

GENERANDO M06: GRAFO PYVIS
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m06_grafo_pyvis.html
✅ m06_grafo_pyvis.html


In [6]:
# ============================================================================
# CELDA 6: DOCUMENTO WORD
# ============================================================================
#
# Genera un DOCX de trazabilidad formal con:
#   - Título y autora
#   - Resumen de la unión
#   - Listado de tablas con filas × columnas
#
# Útil como entregable de documentación del proyecto.
# ============================================================================

if HAS_DOCX:
    print('=' * 60)
    print('GENERANDO WORD')
    print('=' * 60)

    doc = Document()
    doc.add_heading('Trazabilidad de Columnas - Fase 1', 0).alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph('María José Morte | morte@uji.es').alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph()

    doc.add_heading('1. Resumen', level=1)
    doc.add_paragraph(
        f'Unión de {len(TABLAS)} tablas en df_alumno '
        f'({fmt(DF_INFO["filas"])} × {DF_INFO["n_cols"]} columnas)'
    )

    doc.add_heading('2. Tablas de origen', level=1)
    for t, info in sorted(TABLAS.items()):
        doc.add_paragraph(
            f'• {t}: {fmt(info["filas"])} × {info["n_cols"]} columnas',
            style='List Bullet'
        )

    doc.save(RUTA_DOCS / 'trazabilidad_columnas.docx')
    print('✅ trazabilidad_columnas.docx')
else:
    print('⚠️ python-docx no disponible — Word omitido')

⚠️ python-docx no disponible — Word omitido


In [7]:
# ============================================================================
# CELDA 7: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F1-M05/M06 COMPLETADO')
print('=' * 60)
print(f'📁 trazabilidad_fase1.jpg/png')
if HAS_PYVIS:
    print(f'📁 m05_grafo_d3.html')
    print(f'📁 m06_grafo_pyvis.html')
if HAS_DOCX:
    print(f'📁 trazabilidad_columnas.docx')
print('\n🎉 FASE 1 COMPLETA')


✅ F1-M05/M06 COMPLETADO
📁 trazabilidad_fase1.jpg/png
📁 m05_grafo_d3.html
📁 m06_grafo_pyvis.html

🎉 FASE 1 COMPLETA
